# LeetCode 834 — Sum of Distances in Tree bằng Python

- Link: https://leetcode.com/problems/sum-of-distances-in-tree/
- Độ khó: **Hard**
- Chủ đề: Cây, DFS/BFS, Kích thước cây con, Kỹ thuật đổi gốc (rerooting DP)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def sumOfDistancesInTree(self, n: int, edges: List[List[int]]) -> List[int]:
```

## 1. Giới thiệu & mục tiêu học

Cho một cây vô hướng `n` đỉnh. Với **mỗi** đỉnh `i`, tính tổng khoảng cách từ `i` tới mọi đỉnh khác. Cách dễ nhất: BFS từ mỗi đỉnh → `O(n²)`; với `n = 3·10⁴` là gần `10⁹` phép — quá chậm.

Bài này là ví dụ chuẩn mực của **kỹ thuật đổi gốc (rerooting)**: tính đáp án cho **một** gốc bằng một lần DFS, rồi "dời gốc" sang đỉnh kề với chi phí `O(1)` nhờ công thức `ans[con] = ans[cha] − size[con] + (n − size[con])`. Tổng cộng chỉ `O(n)`.

Sau notebook này bạn sẽ:

1. Biểu diễn cây bằng danh sách kề và duyệt BFS/DFS không đệ quy.
2. Tính **kích thước cây con** `size[v]` và tổng độ sâu từ gốc.
3. Hiểu trực giác "dời gốc qua một cạnh": `size[con]` đỉnh gần thêm 1, `n − size[con]` đỉnh xa thêm 1.
4. Cài rerooting hai lượt (hậu thứ tự rồi tiền thứ tự) cả đệ quy lẫn lặp.
5. Nhận ra khuôn mẫu rerooting để áp dụng cho các bài "tính cho mọi gốc".

## 2. Đề bài & ví dụ

**Phát biểu lại.** Có một cây vô hướng liên thông gồm `n` đỉnh đánh số `0..n−1` và `n − 1` cạnh `edges[i] = [a, b]`. Trả về mảng `answer` độ dài `n`, trong đó `answer[i]` = tổng khoảng cách (số cạnh trên đường đi) từ đỉnh `i` tới tất cả các đỉnh khác.

**Ví dụ 1**

```text
Input: n = 6, edges = [[0,1],[0,2],[2,3],[2,4],[2,5]]
Output: [8,12,6,10,10,10]
Giải thích: dist(0,1) + dist(0,2) + dist(0,3) + dist(0,4) + dist(0,5) = 1 + 1 + 2 + 2 + 2 = 8.
```

```text
      0
     / \
    1   2
       /|\
      3 4 5
```

**Ví dụ 2**

```text
Input: n = 1, edges = []
Output: [0]
```

**Ví dụ 3**

```text
Input: n = 2, edges = [[1,0]]
Output: [1,1]
```

**Ràng buộc**

- `1 <= n <= 3·10⁴`
- `edges.length == n − 1`, `0 <= aᵢ, bᵢ < n`, `aᵢ != bᵢ`
- Đầu vào là một cây hợp lệ.

In [1]:
from typing import List

EXAMPLES = [
    (6, [[0, 1], [0, 2], [2, 3], [2, 4], [2, 5]], [8, 12, 6, 10, 10, 10]),
    (1, [], [0]),
    (2, [[1, 0]], [1, 1]),
    (3, [[0, 1], [1, 2]], [3, 2, 3]),             # đường thẳng
    (4, [[0, 1], [0, 2], [0, 3]], [3, 5, 5, 5]),  # hình sao
]
for n, e, exp in EXAMPLES:
    print(n, e, "-> kỳ vọng", exp)

6 [[0, 1], [0, 2], [2, 3], [2, 4], [2, 5]] -> kỳ vọng [8, 12, 6, 10, 10, 10]
1 [] -> kỳ vọng [0]
2 [[1, 0]] -> kỳ vọng [1, 1]
3 [[0, 1], [1, 2]] -> kỳ vọng [3, 2, 3]
4 [[0, 1], [0, 2], [0, 3]] -> kỳ vọng [3, 5, 5, 5]


## 3. Kiến thức nền

### 3.1. Danh sách kề cho cây

Cây vô hướng: mỗi cạnh `[a, b]` thêm `b` vào `adj[a]` và `a` vào `adj[b]`.

In [2]:
def build_adj(n, edges):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b)
        adj[b].append(a)
    return adj

print(build_adj(6, [[0, 1], [0, 2], [2, 3], [2, 4], [2, 5]]))

[[1, 2], [0], [0, 3, 4, 5], [2], [2], [2]]


### 3.2. BFS tính khoảng cách từ một đỉnh

Trên cây (đồ thị không trọng số), BFS cho khoảng cách chính xác tới mọi đỉnh.

In [3]:
from collections import deque

def bfs_dist(adj, src):
    dist = [-1] * len(adj)
    dist[src] = 0
    q = deque([src])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if dist[v] == -1:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

adj = build_adj(6, [[0, 1], [0, 2], [2, 3], [2, 4], [2, 5]])
print(bfs_dist(adj, 0), sum(bfs_dist(adj, 0)))

[0, 1, 1, 2, 2, 2] 8


### 3.3. Thứ tự BFS và mảng cha — thay cho đệ quy

Duyệt BFS từ gốc, ghi `order` (thứ tự thăm) và `parent`. Khi đó:

- Duyệt `order` **ngược** = mọi con được xử lý trước cha (giống hậu thứ tự).
- Duyệt `order` **xuôi** = cha trước con (giống tiền thứ tự).

In [4]:
def bfs_order(adj, root=0):
    parent = [-1] * len(adj)
    order = [root]
    seen = [False] * len(adj)
    seen[root] = True
    for u in order:                       # danh sách lớn dần trong lúc duyệt
        for v in adj[u]:
            if not seen[v]:
                seen[v] = True
                parent[v] = u
                order.append(v)
    return order, parent

print(bfs_order(adj))

([0, 1, 2, 3, 4, 5], [-1, 0, 0, 2, 2, 2])


### 3.4. Kích thước cây con

`size[v] = 1 + Σ size[con]`. Tính bằng cách duyệt `order` ngược, cộng `size[v]` vào `size[parent[v]]`.

In [5]:
def subtree_sizes(adj, root=0):
    order, parent = bfs_order(adj, root)
    size = [1] * len(adj)
    for v in reversed(order):
        if parent[v] != -1:
            size[parent[v]] += size[v]
    return size

print(subtree_sizes(adj))

[6, 1, 4, 1, 1, 1]


### 3.5. Giới hạn đệ quy trong Python

Mặc định ~1000 tầng. Cây đường thẳng 3·10⁴ đỉnh → DFS đệ quy tràn. Có thể `sys.setrecursionlimit` nhưng an toàn hơn là viết lặp (mục 3.3).

In [6]:
import sys
print(sys.getrecursionlimit())

1000


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Tính tay cho gốc 0 (ví dụ 1)

Độ sâu từ 0: `depth = [0, 1, 1, 2, 2, 2]` → `ans[0] = 8`.

Kích thước cây con (gốc 0): `size = [6, 1, 4, 1, 1, 1]`.

### 4.2. Dời gốc từ 0 sang 2

Khi đứng ở 2 thay vì 0:

- 4 đỉnh trong cây con của 2 (`{2, 3, 4, 5}`) **gần hơn 1** → `−4`.
- `6 − 4 = 2` đỉnh còn lại (`{0, 1}`) **xa hơn 1** → `+2`.

`ans[2] = ans[0] − 4 + 2 = 6` ✔ (kiểm tra trực tiếp: `dist(2,0) = 1`, `dist(2,1) = 2`, `dist(2,3) = dist(2,4) = dist(2,5) = 1` → `1 + 2 + 1 + 1 + 1 = 6`).

### 4.3. Tiếp tục

- `ans[1] = ans[0] − size[1] + (n − size[1]) = 8 − 1 + 5 = 12` ✔
- `ans[3] = ans[2] − 1 + 5 = 10` ✔ (tương tự 4, 5)

→ Chỉ cần `ans[gốc]` và `size[]` là suy ra mọi đáp án.

In [7]:
n0, e0 = 6, [[0, 1], [0, 2], [2, 3], [2, 4], [2, 5]]
a0 = build_adj(n0, e0)
print("depth từ 0:", bfs_dist(a0, 0), "| size:", subtree_sizes(a0))
print("ans[2] =", 8 - 4 + (6 - 4), "| ans[1] =", 8 - 1 + 5)

depth từ 0: [0, 1, 1, 2, 2, 2] | size: [6, 1, 4, 1, 1, 1]
ans[2] = 6 | ans[1] = 12


## 5. Cách 1 — Brute Force: BFS từ mọi đỉnh

**Ý tưởng.** Với mỗi đỉnh `i`, BFS tính khoảng cách tới mọi đỉnh và cộng lại.

In [8]:
def sum_dist_brute(n, edges):
    adj = build_adj(n, edges)
    return [sum(bfs_dist(adj, i)) for i in range(n)]

for n, e, exp in EXAMPLES:
    assert sum_dist_brute(n, e) == exp
print("Brute force: OK")

Brute force: OK


**Độ phức tạp.** `n` lần BFS, mỗi lần `O(n)` → `O(n²)` thời gian, `O(n)` bộ nhớ. Với `n = 3·10⁴`: `9·10⁸` thao tác — TLE trong Python.

**Nút thắt.** Các lần BFS từ hai đỉnh kề nhau gần như giống hệt nhau: khoảng cách tới mỗi đỉnh chỉ thay đổi đúng `±1`. Ta lặp lại quá nhiều công việc.

## 6. Bước đột phá

**Insight — Dời gốc qua một cạnh.** Xét cạnh `(u, v)` với `v` là con của `u` (khi gốc tại 0). Cắt cạnh này, cây chia thành hai phần: phần chứa `v` có `size[v]` đỉnh, phần chứa `u` có `n − size[v]` đỉnh. Khi đi từ `u` sang `v`:

- mọi đỉnh trong phần của `v` gần hơn 1;
- mọi đỉnh trong phần của `u` xa hơn 1.

Vậy **`ans[v] = ans[u] − size[v] + (n − size[v]) = ans[u] + n − 2·size[v]`**.

**Kế hoạch hai lượt:**

1. **Lượt 1 (từ dưới lên):** chọn gốc 0, tính `size[v]` và `ans[0] = Σ depth`. Có thể tính `ans[0]` qua `sub[v]` = tổng khoảng cách từ `v` tới các đỉnh trong cây con của `v`: `sub[v] = Σ (sub[c] + size[c])`.
2. **Lượt 2 (từ trên xuống):** theo thứ tự cha trước con, `ans[v] = ans[parent] + n − 2·size[v]`.

**Invariant.** Khi xử lý `v` ở lượt 2, `ans[parent[v]]` đã đúng.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Rerooting bằng DFS đệ quy

Dễ đọc nhất: hai hàm DFS. Cần tăng giới hạn đệ quy cho cây sâu (chỉ để minh hoạ; trên LeetCode cây đường thẳng 3·10⁴ đỉnh có thể tràn stack C).

In [9]:
def sum_dist_recursive(n, edges):
    sys.setrecursionlimit(max(10000, 2 * n + 100))
    adj = build_adj(n, edges)
    size = [1] * n
    ans = [0] * n

    def dfs1(u, p):                 # tính size[u] và sub (lưu tạm vào ans[u])
        for v in adj[u]:
            if v != p:
                dfs1(v, u)
                size[u] += size[v]
                ans[u] += ans[v] + size[v]

    def dfs2(u, p):                 # đổi gốc từ trên xuống
        for v in adj[u]:
            if v != p:
                ans[v] = ans[u] + n - 2 * size[v]
                dfs2(v, u)

    dfs1(0, -1)                     # ans[0] lúc này là đáp án đúng cho gốc 0
    dfs2(0, -1)
    return ans

for n, e, exp in EXAMPLES:
    assert sum_dist_recursive(n, e) == exp
print("Rerooting đệ quy: OK")

Rerooting đệ quy: OK


Giải thích:

- Sau `dfs1`, `ans[u]` tạm chứa `sub[u]` = tổng khoảng cách từ `u` tới các đỉnh **trong cây con** của `u`. Mỗi đỉnh trong cây con của con `v` cách `u` xa hơn cách `v` đúng 1 → `sub[v] + size[v]`.
- Với gốc 0, cây con là cả cây → `ans[0] = sub[0]` là đáp án thật.
- `dfs2` ghi đè `ans[v]` (đang chứa `sub[v]`) bằng đáp án thật — an toàn vì `sub[v]` không còn cần nữa.

Code nộp LeetCode cho cách 2:

In [10]:
class SolutionRecursive:
    def sumOfDistancesInTree(self, n: int, edges: List[List[int]]) -> List[int]:
        sys.setrecursionlimit(max(10000, 2 * n + 100))
        adj = [[] for _ in range(n)]
        for a, b in edges:
            adj[a].append(b)
            adj[b].append(a)
        size, ans = [1] * n, [0] * n

        def dfs1(u, p):
            for v in adj[u]:
                if v != p:
                    dfs1(v, u)
                    size[u] += size[v]
                    ans[u] += ans[v] + size[v]

        def dfs2(u, p):
            for v in adj[u]:
                if v != p:
                    ans[v] = ans[u] + n - 2 * size[v]
                    dfs2(v, u)

        dfs1(0, -1)
        dfs2(0, -1)
        return ans

for n, e, exp in EXAMPLES:
    assert SolutionRecursive().sumOfDistancesInTree(n, e) == exp
print("SolutionRecursive: OK")

SolutionRecursive: OK


**Độ phức tạp cách 2:** `O(n)` thời gian, `O(n)` bộ nhớ (kể cả stack đệ quy sâu tới `n`).

### 7.2. Cách 3 — Rerooting lặp với thứ tự BFS

Thay đệ quy bằng `order`/`parent` (mục 3.3):

1. `order` ngược: cộng `size` và `sub` lên cha.
2. `order` xuôi (bỏ gốc): `ans[v] = ans[parent[v]] + n − 2·size[v]`.

In [11]:
def sum_dist_iterative(n, edges):
    adj = build_adj(n, edges)
    order, parent = bfs_order(adj, 0)
    size = [1] * n
    sub = [0] * n
    for v in reversed(order):
        p = parent[v]
        if p != -1:
            size[p] += size[v]
            sub[p] += sub[v] + size[v]
    ans = [0] * n
    ans[0] = sub[0]
    for v in order[1:]:
        ans[v] = ans[parent[v]] + n - 2 * size[v]
    return ans

for n, e, exp in EXAMPLES:
    assert sum_dist_iterative(n, e) == exp
print("Rerooting lặp: OK")

Rerooting lặp: OK


Lưu ý thứ tự trong vòng ngược: khi xử lý `v`, mọi con của `v` đã nằm **sau** `v` trong `order` nên đã được xử lý → `size[v]`, `sub[v]` đã hoàn chỉnh trước khi cộng lên `p`.

Code nộp LeetCode cho cách 3:

In [12]:
class SolutionIterative:
    def sumOfDistancesInTree(self, n: int, edges: List[List[int]]) -> List[int]:
        adj = [[] for _ in range(n)]
        for a, b in edges:
            adj[a].append(b)
            adj[b].append(a)
        parent = [-1] * n
        order = [0]
        seen = [False] * n
        seen[0] = True
        for u in order:
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    parent[v] = u
                    order.append(v)
        size, sub = [1] * n, [0] * n
        for v in reversed(order):
            p = parent[v]
            if p >= 0:
                size[p] += size[v]
                sub[p] += sub[v] + size[v]
        ans = [0] * n
        ans[0] = sub[0]
        for v in order[1:]:
            ans[v] = ans[parent[v]] + n - 2 * size[v]
        return ans

for n, e, exp in EXAMPLES:
    assert SolutionIterative().sumOfDistancesInTree(n, e) == exp
print("SolutionIterative: OK")

SolutionIterative: OK


**Độ phức tạp cách 3:** `O(n)` thời gian, `O(n)` bộ nhớ, không đệ quy.

### 7.3. Cách 4 — `ans[0]` bằng tổng độ sâu, rồi đổi gốc

Biến thể đơn giản hoá lượt 1: `ans[0] = Σ depth[v]` (một BFS), `size` như trước. Không cần mảng `sub`.

In [13]:
def sum_dist_depth(n, edges):
    adj = build_adj(n, edges)
    order, parent = bfs_order(adj, 0)
    depth = [0] * n
    for v in order[1:]:
        depth[v] = depth[parent[v]] + 1
    size = [1] * n
    for v in reversed(order):
        if parent[v] >= 0:
            size[parent[v]] += size[v]
    ans = [0] * n
    ans[0] = sum(depth)
    for v in order[1:]:
        ans[v] = ans[parent[v]] + n - 2 * size[v]
    return ans

for n, e, exp in EXAMPLES:
    assert sum_dist_depth(n, e) == exp
print("Tổng độ sâu + đổi gốc: OK")

Tổng độ sâu + đổi gốc: OK


### 7.4. Cách 5 — Đóng góp theo cạnh (kiểm chứng tổng)

Một hệ quả hay: **tổng tất cả** `ans[i]` = `2 · Σ_{cạnh (u,v)} size[v]·(n − size[v])` (mỗi cạnh nằm trên đường đi của đúng `size·(n−size)` cặp không thứ tự, mỗi cặp được đếm hai lần trong `Σ ans`). Không cho từng `ans[i]` nhưng là cách kiểm tra rất tốt, và là lời giải trực tiếp cho bài "tổng khoảng cách mọi cặp".

In [14]:
def total_pair_distance(n, edges):
    adj = build_adj(n, edges)
    order, parent = bfs_order(adj, 0)
    size = [1] * n
    for v in reversed(order):
        if parent[v] >= 0:
            size[parent[v]] += size[v]
    return sum(size[v] * (n - size[v]) for v in order[1:])

for n, e, exp in EXAMPLES:
    assert 2 * total_pair_distance(n, e) == sum(exp)
print("Kiểm chứng tổng theo cạnh: OK")

Kiểm chứng tổng theo cạnh: OK


Code nộp LeetCode cho cách 4 (gọn nhất trong các bản lặp):

In [15]:
class SolutionDepth:
    def sumOfDistancesInTree(self, n: int, edges: List[List[int]]) -> List[int]:
        adj = [[] for _ in range(n)]
        for a, b in edges:
            adj[a].append(b)
            adj[b].append(a)
        parent, depth = [-1] * n, [0] * n
        order, seen = [0], [False] * n
        seen[0] = True
        for u in order:
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    parent[v] = u
                    depth[v] = depth[u] + 1
                    order.append(v)
        size = [1] * n
        for v in reversed(order):
            if parent[v] >= 0:
                size[parent[v]] += size[v]
        ans = [0] * n
        ans[0] = sum(depth)
        for v in order[1:]:
            ans[v] = ans[parent[v]] + n - 2 * size[v]
        return ans

for n, e, exp in EXAMPLES:
    assert SolutionDepth().sumOfDistancesInTree(n, e) == exp
print("SolutionDepth: OK")

SolutionDepth: OK


**Độ phức tạp cách 4:** `O(n)` / `O(n)`. Cách 5 (`O(n)`) chỉ cho tổng, dùng để kiểm tra.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1: in `order`, `parent`, rồi lượt 1 (từ dưới lên) và lượt 2 (từ trên xuống).

In [16]:
def trace_reroot(n, edges):
    adj = build_adj(n, edges)
    order, parent = bfs_order(adj, 0)
    print("order :", order)
    print("parent:", parent)
    size, sub = [1] * n, [0] * n
    print("--- Lượt 1: duyệt order ngược ---")
    for v in reversed(order):
        p = parent[v]
        if p != -1:
            size[p] += size[v]
            sub[p] += sub[v] + size[v]
            print(f"  v={v}: size[{v}]={size[v]}, sub[{v}]={sub[v]} -> cộng lên cha {p}: size[{p}]={size[p]}, sub[{p}]={sub[p]}")
    ans = [0] * n
    ans[0] = sub[0]
    print(f"ans[0] = sub[0] = {ans[0]}")
    print("--- Lượt 2: duyệt order xuôi ---")
    for v in order[1:]:
        p = parent[v]
        ans[v] = ans[p] + n - 2 * size[v]
        print(f"  ans[{v}] = ans[{p}] + n - 2*size[{v}] = {ans[p]} + {n} - {2 * size[v]} = {ans[v]}")
    return ans

trace_reroot(6, [[0, 1], [0, 2], [2, 3], [2, 4], [2, 5]])

order : [0, 1, 2, 3, 4, 5]
parent: [-1, 0, 0, 2, 2, 2]
--- Lượt 1: duyệt order ngược ---
  v=5: size[5]=1, sub[5]=0 -> cộng lên cha 2: size[2]=2, sub[2]=1
  v=4: size[4]=1, sub[4]=0 -> cộng lên cha 2: size[2]=3, sub[2]=2
  v=3: size[3]=1, sub[3]=0 -> cộng lên cha 2: size[2]=4, sub[2]=3
  v=2: size[2]=4, sub[2]=3 -> cộng lên cha 0: size[0]=5, sub[0]=7
  v=1: size[1]=1, sub[1]=0 -> cộng lên cha 0: size[0]=6, sub[0]=8
ans[0] = sub[0] = 8
--- Lượt 2: duyệt order xuôi ---
  ans[1] = ans[0] + n - 2*size[1] = 8 + 6 - 2 = 12
  ans[2] = ans[0] + n - 2*size[2] = 8 + 6 - 8 = 6
  ans[3] = ans[2] + n - 2*size[3] = 6 + 6 - 2 = 10
  ans[4] = ans[2] + n - 2*size[4] = 6 + 6 - 2 = 10
  ans[5] = ans[2] + n - 2*size[5] = 6 + 6 - 2 = 10


[8, 12, 6, 10, 10, 10]

Bảng trạng thái:

| đỉnh | parent | size | sub (lượt 1) | ans (lượt 2) |
|---|---|---|---|---|
| 0 | −1 | 6 | 8 | 8 |
| 1 | 0 | 1 | 0 | 8 + 6 − 2 = 12 |
| 2 | 0 | 4 | 3 | 8 + 6 − 8 = 6 |
| 3 | 2 | 1 | 0 | 6 + 6 − 2 = 10 |
| 4 | 2 | 1 | 0 | 10 |
| 5 | 2 | 1 | 0 | 10 |

Trace thêm một cây đường thẳng 5 đỉnh `0-1-2-3-4` để thấy `ans` giảm dần về giữa rồi tăng:

In [17]:
trace_reroot(5, [[0, 1], [1, 2], [2, 3], [3, 4]])

order : [0, 1, 2, 3, 4]
parent: [-1, 0, 1, 2, 3]
--- Lượt 1: duyệt order ngược ---
  v=4: size[4]=1, sub[4]=0 -> cộng lên cha 3: size[3]=2, sub[3]=1
  v=3: size[3]=2, sub[3]=1 -> cộng lên cha 2: size[2]=3, sub[2]=3
  v=2: size[2]=3, sub[2]=3 -> cộng lên cha 1: size[1]=4, sub[1]=6
  v=1: size[1]=4, sub[1]=6 -> cộng lên cha 0: size[0]=5, sub[0]=10
ans[0] = sub[0] = 10
--- Lượt 2: duyệt order xuôi ---
  ans[1] = ans[0] + n - 2*size[1] = 10 + 5 - 8 = 7
  ans[2] = ans[1] + n - 2*size[2] = 7 + 5 - 6 = 6
  ans[3] = ans[2] + n - 2*size[3] = 6 + 5 - 4 = 7
  ans[4] = ans[3] + n - 2*size[4] = 7 + 5 - 2 = 10


[10, 7, 6, 7, 10]

## 9. Code nộp LeetCode

Chọn cách 3 (rerooting lặp với `sub`): không đệ quy, `O(n)`.

In [18]:
from typing import List

class Solution:
    def sumOfDistancesInTree(self, n: int, edges: List[List[int]]) -> List[int]:
        adj = [[] for _ in range(n)]
        for a, b in edges:
            adj[a].append(b)
            adj[b].append(a)
        # Thứ tự BFS từ gốc 0 + mảng cha
        parent = [-1] * n
        order = [0]
        seen = [False] * n
        seen[0] = True
        for u in order:
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    parent[v] = u
                    order.append(v)
        # Lượt 1 (con trước cha): kích thước cây con và tổng khoảng cách trong cây con
        size = [1] * n
        sub = [0] * n
        for v in reversed(order):
            p = parent[v]
            if p >= 0:
                size[p] += size[v]
                sub[p] += sub[v] + size[v]
        # Lượt 2 (cha trước con): đổi gốc
        ans = [0] * n
        ans[0] = sub[0]
        for v in order[1:]:
            ans[v] = ans[parent[v]] + n - 2 * size[v]
        return ans

for n, e, exp in EXAMPLES:
    assert Solution().sumOfDistancesInTree(n, e) == exp
print("Solution: OK")

Solution: OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (lượt 1).** Với gốc 0, `sub[v] = Σ_{w ∈ T(v)} dist(v, w)` trong đó `T(v)` là cây con của `v`. Chứng minh quy nạp từ lá lên: lá có `sub = 0`. Với `v` có các con `c`, mỗi `w ∈ T(c)` có `dist(v, w) = dist(c, w) + 1` (đường đi duy nhất qua `c`). Cộng lại: `sub[v] = Σ_c (sub[c] + size[c])`. Vậy `sub[0]` = tổng khoảng cách từ 0 tới mọi đỉnh = `ans[0]`.

**Bổ đề 2 (đổi gốc).** Cho cạnh `(p, v)`, `v` là con của `p`. Với `w ∈ T(v)`: đường từ `p` tới `w` đi qua `v` → `dist(v, w) = dist(p, w) − 1`. Với `w ∉ T(v)`: đường từ `v` tới `w` đi qua `p` → `dist(v, w) = dist(p, w) + 1`. Cộng trên mọi `w`: `ans[v] = ans[p] − |T(v)| + (n − |T(v)|) = ans[p] + n − 2·size[v]`.

**Thứ tự.** Lượt 2 duyệt theo thứ tự BFS: cha luôn đứng trước con, nên `ans[p]` đã đúng khi tính `ans[v]`. Quy nạp theo độ sâu → mọi `ans` đúng. ∎

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. BFS từ mọi đỉnh | `O(n²)` | `O(n)` |
| 2. Rerooting đệ quy | `O(n)` | `O(n)` + stack đệ quy |
| 3. Rerooting lặp (`sub`) | `O(n)` | `O(n)` |
| 4. Tổng độ sâu + đổi gốc | `O(n)` | `O(n)` |
| 5. Đóng góp cạnh (chỉ tổng) | `O(n)` | `O(n)` |

In [19]:
import random, time

def random_tree(n, rnd):
    return [[i, rnd.randrange(i)] for i in range(1, n)]

rnd = random.Random(834)
big_n = 30000
big_e = random_tree(big_n, rnd)
line_e = [[i, i + 1] for i in range(big_n - 1)]          # cây sâu nhất
for name, fn in [("iterative", sum_dist_iterative), ("depth", sum_dist_depth),
                 ("Solution", lambda n, e: Solution().sumOfDistancesInTree(n, e))]:
    t0 = time.perf_counter()
    r1 = fn(big_n, big_e); r2 = fn(big_n, line_e)
    print(f"{name:10}: ngẫu nhiên ans[0]={r1[0]}, đường thẳng ans[0]={r2[0]}  ({(time.perf_counter() - t0) * 1000:.0f} ms)")

t0 = time.perf_counter(); sum_dist_brute(2000, random_tree(2000, rnd)); print(f"brute n=2000: {(time.perf_counter() - t0) * 1000:.0f} ms")

iterative : ngẫu nhiên ans[0]=290711, đường thẳng ans[0]=449985000  (36 ms)


depth     : ngẫu nhiên ans[0]=290711, đường thẳng ans[0]=449985000  (76 ms)


Solution  : ngẫu nhiên ans[0]=290711, đường thẳng ans[0]=449985000  (44 ms)


brute n=2000: 691 ms


## 12. Bộ test

In [20]:
methods = {
    "brute": sum_dist_brute,
    "recursive": sum_dist_recursive,
    "iterative": sum_dist_iterative,
    "depth": sum_dist_depth,
    "SolutionRecursive": lambda n, e: SolutionRecursive().sumOfDistancesInTree(n, e),
    "SolutionIterative": lambda n, e: SolutionIterative().sumOfDistancesInTree(n, e),
    "SolutionDepth": lambda n, e: SolutionDepth().sumOfDistancesInTree(n, e),
    "Solution": lambda n, e: Solution().sumOfDistancesInTree(n, e),
}

# 12.1 Ví dụ + edge cases
edge = EXAMPLES + [
    (5, [[0, 1], [1, 2], [2, 3], [3, 4]], [10, 7, 6, 7, 10]),
    (5, [[4, 3], [3, 2], [2, 1], [1, 0]], [10, 7, 6, 7, 10]),    # cạnh ngược chiều
    (7, [[0, 1], [0, 2], [1, 3], [1, 4], [2, 5], [2, 6]], [10, 11, 11, 16, 16, 16, 16]),
]
for name, fn in methods.items():
    for n, e, exp in edge:
        assert fn(n, e) == exp, (name, n, e, fn(n, e), exp)
print("Ví dụ + edge cases: OK")

# 12.2 Ngẫu nhiên có seed: đối chiếu brute force (đánh số đỉnh xáo trộn)
for seed in range(400):
    r_ = random.Random(seed)
    n = r_.randint(1, 30)
    perm = list(range(n)); r_.shuffle(perm)
    e = [[perm[a], perm[b]] for a, b in random_tree(n, r_)]
    r_.shuffle(e)
    ref = sum_dist_brute(n, e)
    for name, fn in methods.items():
        assert fn(n, e) == ref, (name, n, e)
    assert 2 * total_pair_distance(n, e) == sum(ref)
print("400 cây ngẫu nhiên: khớp brute force")

Ví dụ + edge cases: OK
400 cây ngẫu nhiên: khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Sai dấu trong công thức đổi gốc

**Code sai:**

In [21]:
def wrong_sign(n, edges):
    adj = build_adj(n, edges)
    order, parent = bfs_order(adj, 0)
    size, sub = [1] * n, [0] * n
    for v in reversed(order):
        p = parent[v]
        if p >= 0:
            size[p] += size[v]; sub[p] += sub[v] + size[v]
    ans = [0] * n; ans[0] = sub[0]
    for v in order[1:]:
        ans[v] = ans[parent[v]] - n + 2 * size[v]          # SAI dấu
    return ans

print("sai:", wrong_sign(6, EXAMPLES[0][1]), " đúng:", EXAMPLES[0][2])

sai: [8, 4, 10, 6, 6, 6]  đúng: [8, 12, 6, 10, 10, 10]


**Vì sao sai:** sang con `v`, các đỉnh trong cây con `v` **gần hơn** (trừ `size[v]`), các đỉnh còn lại **xa hơn** (cộng `n − size[v]`). **Code đúng:** `ans[v] = ans[p] + n − 2·size[v]`.

### Lỗi 2 — Lượt 1 cộng `sub[v]` mà quên `+ size[v]`

**Code sai:**

In [22]:
def wrong_sub(n, edges):
    adj = build_adj(n, edges)
    order, parent = bfs_order(adj, 0)
    size, sub = [1] * n, [0] * n
    for v in reversed(order):
        p = parent[v]
        if p >= 0:
            size[p] += size[v]
            sub[p] += sub[v]                                 # SAI: thiếu + size[v]
    ans = [0] * n; ans[0] = sub[0]
    for v in order[1:]:
        ans[v] = ans[parent[v]] + n - 2 * size[v]
    return ans

print("sai:", wrong_sub(6, EXAMPLES[0][1]), " đúng:", EXAMPLES[0][2])

sai: [0, 4, -2, 2, 2, 2]  đúng: [8, 12, 6, 10, 10, 10]


**Vì sao sai:** mỗi đỉnh trong cây con của `v` xa `p` hơn xa `v` đúng 1 cạnh — có `size[v]` đỉnh như vậy. Thiếu số hạng này, `sub` luôn bằng 0.

### Lỗi 3 — Lượt 2 duyệt sai thứ tự (con trước cha)

**Code sai:**

In [23]:
def wrong_order(n, edges):
    adj = build_adj(n, edges)
    order, parent = bfs_order(adj, 0)
    size, sub = [1] * n, [0] * n
    for v in reversed(order):
        p = parent[v]
        if p >= 0:
            size[p] += size[v]; sub[p] += sub[v] + size[v]
    ans = [0] * n; ans[0] = sub[0]
    for v in reversed(order[1:]):                            # SAI: con trước cha
        ans[v] = ans[parent[v]] + n - 2 * size[v]
    return ans

print("sai:", wrong_order(6, EXAMPLES[0][1]), " đúng:", EXAMPLES[0][2])

sai: [8, 12, 6, 4, 4, 4]  đúng: [8, 12, 6, 10, 10, 10]


**Vì sao sai:** `ans[3]` được tính từ `ans[2]` khi `ans[2]` còn là 0. **Code đúng:** duyệt `order` xuôi (tiền thứ tự / BFS).

### Lỗi 4 — DFS đệ quy không tăng giới hạn trên cây sâu

Cây đường thẳng 3·10⁴ đỉnh → `RecursionError`. Ngay cả khi tăng `setrecursionlimit`, stack C có thể tràn (crash). Dùng bản lặp.

In [24]:
def dfs_depth_naive(adj, u, p):
    best = 0
    for v in adj[u]:
        if v != p:
            best = max(best, 1 + dfs_depth_naive(adj, v, u))
    return best

old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    dfs_depth_naive(build_adj(5000, [[i, i + 1] for i in range(4999)]), 0, -1)
    print("không lỗi?!")
except RecursionError as ex:
    print("RecursionError:", str(ex)[:40])
finally:
    sys.setrecursionlimit(max(old, 10000))

RecursionError: maximum recursion depth exceeded


## 14. Mẹo & nhận dạng pattern

- **"Tính một đại lượng cho mọi đỉnh làm gốc"** trên cây → **rerooting**: một DFS từ gốc cố định + công thức chuyển cha → con `O(1)`.
- Công thức chuyển thường dùng **kích thước cây con** `size[v]` và `n − size[v]` (hai phía của cạnh).
- **Tránh đệ quy** với cây lớn: dùng thứ tự BFS + `parent`, duyệt ngược để gom từ dưới lên, xuôi để đẩy từ trên xuống.
- **Đóng góp theo cạnh**: cạnh `(p, v)` nằm trên `size[v]·(n − size[v])` đường đi.
- Bài liên quan: **310 Minimum Height Trees**, **2581 Count Number of Possible Root Nodes**, **2858 Minimum Edge Reversals So Every Node Is Reachable**, **2477 Minimum Fuel Cost**, **1617**.

## 15. Biến thể

1. **Cạnh có trọng số `w`**: `sub[p] += sub[v] + size[v]·w`, `ans[v] = ans[p] + w·(n − 2·size[v])`.
2. **Đỉnh có trọng số (số người ở mỗi đỉnh)**: thay `size` bằng tổng trọng số cây con, `n` bằng tổng trọng số.
3. **Tìm đỉnh có tổng khoảng cách nhỏ nhất** (trung vị của cây): `argmin(ans)` — luôn là **trọng tâm** (centroid).
4. **Tổng khoảng cách mọi cặp**: `Σ size[v]·(n − size[v])` (cách 5).
5. **Tổng bình phương khoảng cách**: rerooting với thêm đại lượng (tổng khoảng cách và tổng bình phương), công thức chuyển dùng `(d ± 1)² = d² ± 2d + 1`.

Minh hoạ biến thể 1 (cạnh có trọng số), đối chiếu Dijkstra/BFS có trọng số:

In [25]:
def sum_dist_weighted(n, edges):
    adj = [[] for _ in range(n)]
    for a, b, w in edges:
        adj[a].append((b, w)); adj[b].append((a, w))
    parent, pw, order, seen = [-1] * n, [0] * n, [0], [False] * n
    seen[0] = True
    for u in order:
        for v, w in adj[u]:
            if not seen[v]:
                seen[v] = True; parent[v] = u; pw[v] = w; order.append(v)
    size, sub = [1] * n, [0] * n
    for v in reversed(order):
        p = parent[v]
        if p >= 0:
            size[p] += size[v]; sub[p] += sub[v] + size[v] * pw[v]
    ans = [0] * n; ans[0] = sub[0]
    for v in order[1:]:
        ans[v] = ans[parent[v]] + pw[v] * (n - 2 * size[v])
    return ans

def sum_dist_weighted_brute(n, edges):
    adj = [[] for _ in range(n)]
    for a, b, w in edges:
        adj[a].append((b, w)); adj[b].append((a, w))
    res = []
    for s in range(n):
        dist = {s: 0}; st = [s]
        while st:
            u = st.pop()
            for v, w in adj[u]:
                if v not in dist:
                    dist[v] = dist[u] + w; st.append(v)
        res.append(sum(dist.values()))
    return res

for seed in range(100):
    r_ = random.Random(seed)
    n = r_.randint(1, 15)
    e = [[a, b, r_.randint(1, 9)] for a, b in random_tree(n, r_)]
    assert sum_dist_weighted(n, e) == sum_dist_weighted_brute(n, e)
print("Biến thể có trọng số: OK", sum_dist_weighted(3, [[0, 1, 5], [1, 2, 2]]))

Biến thể có trọng số: OK [12, 7, 9]


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS mọi đỉnh | tính trực tiếp | `O(n²)` | `O(n)` | đối chiếu, `n ≤ 2000` |
| 2. Rerooting đệ quy | 2 DFS, công thức đổi gốc | `O(n)` | `O(n)` + stack | cây nông, code ngắn |
| 3. Rerooting lặp (`sub`) | thứ tự BFS ngược/xuôi | `O(n)` | `O(n)` | **nộp bài** |
| 4. Tổng độ sâu + đổi gốc | `ans[0] = Σ depth` | `O(n)` | `O(n)` | đơn giản nhất về lượt 1 |
| 5. Đóng góp theo cạnh | `size·(n−size)` | `O(n)` | `O(n)` | chỉ cần tổng mọi cặp / kiểm tra |

## 17. Bài tập

**Bài 1 (đoán output).** Cây sao 5 đỉnh (tâm 0 nối 1..4): `answer`?

**Bài 2 (tính tay).** Cây nhị phân đầy đủ 7 đỉnh (`0` gốc; `1, 2` con của 0; `3, 4` con của 1; `5, 6` con của 2). Tính `size`, `ans[0]`, rồi đổi gốc để ra mọi `ans`.

**Bài 3 (điền TODO).** Viết `tree_centroid_by_distance(n, edges)` trả về các đỉnh có tổng khoảng cách nhỏ nhất.

**Bài 4 (tự viết lại).** Viết lại `Solution` dùng **DFS bằng stack** (thay vì BFS) để lấy thứ tự tiền thứ tự.

**Bài 5 (mini challenge).** Tính `Σ_j dist(i, j)²` cho mọi `i` trong `O(n)` (biến thể 5).

In [26]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def tree_centroid_by_distance_todo(n, edges):
    # TODO: ans = Solution().sumOfDistancesInTree(n, edges); trả về các i có ans[i] nhỏ nhất
    pass

# Bài 4 — TODO
def solution_stack_todo(n, edges):
    # TODO: stack = [0]; khi pop u thì append u vào order; push các con chưa thăm
    pass

# Bài 5 — TODO
def sum_sq_dist_todo(n, edges):
    # TODO: lượt 1: size, s1 (tổng khoảng cách trong cây con), s2 (tổng bình phương)
    # lượt 2: chuyển A1, A2 từ cha sang con bằng (d-1)^2 và (d+1)^2
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: tâm cách mọi đỉnh 1.
- Mức 2: lá cách tâm 1, cách 3 lá khác 2.
- Mức 3: `[4, 7, 7, 7, 7]`.

**Bài 2.**
- Mức 1: `size = [7, 3, 3, 1, 1, 1, 1]`, `ans[0] = 1+1+2+2+2+2 = 10`.
- Mức 2: `ans[1] = 10 + 7 − 6 = 11`, `ans[3] = 11 + 7 − 2 = 16`.
- Mức 3: `[10, 11, 11, 16, 16, 16, 16]`.

**Bài 3.**
- Mức 1: dùng `Solution`.
- Mức 2: `m = min(ans)`.
- Mức 3: luôn có 1 hoặc 2 đỉnh (2 đỉnh thì kề nhau).

**Bài 4.**
- Mức 1: `stack = [0]`, `seen[0] = True`.
- Mức 2: thứ tự pop là một tiền thứ tự hợp lệ (cha pop trước con).
- Mức 3: phần còn lại giữ nguyên.

**Bài 5.**
- Mức 1: lượt 1: `s1[p] += s1[v] + size[v]`, `s2[p] += s2[v] + 2·s1[v] + size[v]`.
- Mức 2: lượt 2: `A1[v] = A1[p] + n − 2·size[v]`. Với `A2`: đỉnh trong cây con `v` có `(d−1)²`, ngoài có `(d+1)²`.
- Mức 3: `A2[v] = A2[p] − 2·(tổng d trong T(v) tính từ p) + size[v] + 2·(tổng d ngoài T(v) tính từ p) + (n − size[v])`, với tổng `d` trong `T(v)` từ `p` = `s1[v] + size[v]`.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
assert Solution().sumOfDistancesInTree(5, [[0, 1], [0, 2], [0, 3], [0, 4]]) == [4, 7, 7, 7, 7]

# Bài 2
assert Solution().sumOfDistancesInTree(7, [[0, 1], [0, 2], [1, 3], [1, 4], [2, 5], [2, 6]]) == [10, 11, 11, 16, 16, 16, 16]

# Bài 3
def tree_centroid_by_distance(n, edges):
    ans = Solution().sumOfDistancesInTree(n, edges)
    m = min(ans)
    return [i for i in range(n) if ans[i] == m]

print(tree_centroid_by_distance(6, EXAMPLES[0][1]), tree_centroid_by_distance(4, [[0, 1], [1, 2], [2, 3]]))

# Bài 4
def solution_stack(n, edges):
    adj = build_adj(n, edges)
    parent, order, seen = [-1] * n, [], [False] * n
    stack = [0]; seen[0] = True
    while stack:
        u = stack.pop()
        order.append(u)
        for v in adj[u]:
            if not seen[v]:
                seen[v] = True; parent[v] = u; stack.append(v)
    size, sub = [1] * n, [0] * n
    for v in reversed(order):
        p = parent[v]
        if p >= 0:
            size[p] += size[v]; sub[p] += sub[v] + size[v]
    ans = [0] * n; ans[0] = sub[0]
    for v in order[1:]:
        ans[v] = ans[parent[v]] + n - 2 * size[v]
    return ans

# Bài 5
def sum_sq_dist(n, edges):
    adj = build_adj(n, edges)
    order, parent = bfs_order(adj, 0)
    size, s1, s2 = [1] * n, [0] * n, [0] * n
    for v in reversed(order):
        p = parent[v]
        if p >= 0:
            size[p] += size[v]
            s2[p] += s2[v] + 2 * s1[v] + size[v]
            s1[p] += s1[v] + size[v]
    A1, A2 = [0] * n, [0] * n
    A1[0], A2[0] = s1[0], s2[0]
    for v in order[1:]:
        p = parent[v]
        inside = s1[v] + size[v]                 # tổng khoảng cách từ p tới đỉnh trong T(v)
        outside = A1[p] - inside                 # từ p tới đỉnh ngoài T(v)
        A2[v] = A2[p] - 2 * inside + size[v] + 2 * outside + (n - size[v])
        A1[v] = A1[p] + n - 2 * size[v]
    return A2

def sum_sq_brute(n, edges):
    adj = build_adj(n, edges)
    return [sum(d * d for d in bfs_dist(adj, i)) for i in range(n)]

for seed in range(200):
    r_ = random.Random(seed)
    n = r_.randint(1, 20)
    e = random_tree(n, r_)
    assert solution_stack(n, e) == sum_dist_brute(n, e)
    assert sum_sq_dist(n, e) == sum_sq_brute(n, e)
print("Tất cả đáp án đúng")

[2] [1, 2]
Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template rerooting (lặp):**

```python
order, parent = bfs_order(adj, 0)        # cha trước con
size = [1]*n; sub = [0]*n
for v in reversed(order):                # con trước cha
    p = parent[v]
    if p >= 0:
        size[p] += size[v]
        sub[p] += sub[v] + size[v]
ans = [0]*n; ans[0] = sub[0]
for v in order[1:]:                      # cha trước con
    ans[v] = ans[parent[v]] + n - 2*size[v]
```

**Checklist trước khi submit:**

- [ ] Danh sách kề **hai chiều**.
- [ ] Lượt 1: `sub[p] += sub[v] + size[v]` (đừng quên `+ size[v]`).
- [ ] Lượt 2: `ans[v] = ans[p] + n − 2·size[v]`, duyệt **cha trước con**.
- [ ] `n = 1` → `[0]` (vòng lặp rỗng tự xử lý).
- [ ] Không đệ quy sâu (cây đường thẳng 3·10⁴).